# 01 — Confirmer Bronze
**Objectif :** vérifier que Bronze est sain avant de construire Silver (étape 1 du plan).
**Tu dois obtenir** 4 confirmations : 8 comptages OK, 8 tables en texte, notes de 1 à 5, 789 `review_id` en double.
**Comment :** exécute les cellules dans l'ordre (Run all).

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
# Les tables Bronze sont supposées s'appeler : orders, customers, order_items, order_payments,
# order_reviews, products, sellers, product_category_name_translation.
# Si les tiennes ont d'autres noms, change-les dans les requêtes (Ctrl+F).

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    """Exécute une requête qui renvoie une seule valeur."""
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    """Compare ce qu'on obtient à ce qu'on attend (valeurs de ton document de travail)."""
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

## 1. Les 8 comptages de lignes

In [0]:
attendus = {
    "orders": 99441, "customers": 99441, "order_items": 112650, "order_payments": 103886,
    "order_reviews": 99224, "products": 32951, "sellers": 3095,
    "category_translation": 73,
}
for table, attendu in attendus.items():
    check(f"lignes {table}", scalar(f"SELECT COUNT(*) FROM {BRONZE}.{table}"), attendu)

✅ OK     lignes orders : obtenu = 99441 | attendu = 99441
✅ OK     lignes customers : obtenu = 99441 | attendu = 99441
✅ OK     lignes order_items : obtenu = 112650 | attendu = 112650
✅ OK     lignes order_payments : obtenu = 103886 | attendu = 103886
✅ OK     lignes order_reviews : obtenu = 99224 | attendu = 99224
✅ OK     lignes products : obtenu = 32951 | attendu = 32951
✅ OK     lignes sellers : obtenu = 3095 | attendu = 3095
❌ ÉCART  lignes category_translation : obtenu = 71 | attendu = 73


In [0]:
spark.sql("SHOW TABLES IN workspace.bronze").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|bronze  |category_translation|false      |
|bronze  |customers           |false      |
|bronze  |order_items         |false      |
|bronze  |order_payments      |false      |
|bronze  |order_reviews       |false      |
|bronze  |orders              |false      |
|bronze  |products            |false      |
|bronze  |sellers             |false      |
+--------+--------------------+-----------+



## 2. Les colonnes sont toutes en texte (string)

In [0]:
for table in attendus:
    types = {f.dataType.simpleString() for f in spark.table(f"{BRONZE}.{table}").schema.fields
             if not f.name.startswith("_")}
    print(("✅ OK     " if types == {"string"} else "❌ ÉCART  ") + f"{table} : types = {types}")

✅ OK     orders : types = {'string'}
✅ OK     customers : types = {'string'}
✅ OK     order_items : types = {'string'}
✅ OK     order_payments : types = {'string'}
✅ OK     order_reviews : types = {'string'}
✅ OK     products : types = {'string'}
✅ OK     sellers : types = {'string'}
✅ OK     category_translation : types = {'string'}


## 3. Les notes d'avis vont de 1 à 5

In [0]:
spark.sql(f"""
    SELECT review_score, COUNT(*) AS nb
    FROM {BRONZE}.order_reviews GROUP BY review_score ORDER BY review_score
""").show()

+------------+-----+
|review_score|   nb|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19142|
|           5|57328|
+------------+-----+



## 4. Les 789 `review_id` en double

In [0]:
nb = scalar(f"""
    SELECT COUNT(*) FROM (
        SELECT review_id FROM {BRONZE}.order_reviews GROUP BY review_id HAVING COUNT(*) > 1)
""")
check("review_id en double", nb, 789)

✅ OK     review_id en double : obtenu = 789 | attendu = 789


## Résultat
Si tout est ✅ (ou si tu comprends chaque ❌), passe au notebook `02_silver_orders`.
Sinon, copie-moi les lignes ❌ et on corrige ensemble.